## Imports

In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import numpy as np
from os import walk

from sklearn.preprocessing import MinMaxScaler, RobustScaler
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error
)

pd.set_option('display.max_columns', None)

from src.get_tables import tickers
from src.model_functions import build_lstm_sequences, split_temporal, generate_lstm_model, get_study_optuna

## Funções

In [3]:
save_folder_path_refined = 'dados/refined'

features = [
    'close', 'high', 'low', 'open', 'volume',
    'close_dolar', 'close_ibovespa', 'close_sp_500', 'selic', 'ipca',
    'ma20', 'ma50', 'bb_upper', 'bb_lower', 'rsi_wilder', 'macd',
    'macd_signal', 'weekday_sin', 'weekday_cos', 'month_sin', 'month_cos'
]

In [38]:
tickers

['RENT3', 'LREN3', 'SMFT3', 'MULT3', 'VBBR3', 'ABEV3']

In [ ]:
for t in tickers:

    tabela_hiperparams = pd.DataFrame()

    df = pd.read_parquet(f"{save_folder_path_refined}/tb_analitica_{t}.parquet", engine = 'pyarrow')
    df = df[df['close'].notna()]
    df = df.sort_values("date").reset_index(drop=True)

    data = df[features].copy()

    # Divisão temporal ANTES do scaler (evita data leakage)
    n = len(data)
    train_end = int(n * 0.70)
    valid_end = int(n * 0.85)

    train_df = data.iloc[:train_end]
    valid_df = data.iloc[train_end:valid_end]
    test_df = data.iloc[valid_end:]

    scaler = MinMaxScaler()
    scaler.fit(train_df)

    scaled_data = np.vstack([
        scaler.transform(train_df),
        scaler.transform(valid_df),
        scaler.transform(test_df)
    ])

    for w in [60, 90, 120, 180]:
        print("=" * 60)
        print(f"Início dos estudos | Ticker = {t} | window_size = {w}")
        print("=" * 60)

        study = get_study_optuna(w, scaled_data, n_trials = 20, ticker = t)

        tmp = pd.DataFrame(data = {
            'ticker':[t],
            'window_size':[w],
            'val_loss':[study.best_value]

        })

        for k, v in study.best_params.items():
            tmp[k] = [v]

        tabela_hiperparams = pd.concat([tabela_hiperparams, tmp])
        print("\n\n")

    tabela_hiperparams.to_excel(f'dados/aux_data/tabela_hiperparametros_{t}.xlsx', index = False)

[I 2026-07-06 19:57:56,647] A new study created in memory with name: LSTM Stock Prediction | window_size = 60


Início dos estudos | Ticker = SMFT3 | window_size = 60
(1157, 60, 21)
(1157,)
Treino     : (809, 60, 21)
Validação  : (174, 60, 21)
Teste      : (174, 60, 21)


  0%|          | 0/20 [00:00<?, ?it/s]

g:\Meu Drive\5. Cursos\Pós ML Engineering\Fase 4 - Deep Learning e IA\lstm-stock-predictor-api\.venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


[I 2026-07-06 19:59:31,693] Trial 0 finished with value: 0.11379677057266235 and parameters: {'units_1': 64, 'units_2': 16, 'dropout': 0.3103085127176983, 'learning_rate': 0.0001245083489731155, 'batch_size': 32}. Best is trial 0 with value: 0.11379677057266235.
[I 2026-07-06 20:00:51,498] Trial 1 finished with value: 0.13822369277477264 and parameters: {'units_1': 64, 'units_2': 64, 'dropout': 0.3576303723549862, 'learning_rate': 0.00021057317087910492, 'batch_size': 64}. Best is trial 0 with value: 0.11379677057266235.
[I 2026-07-06 20:01:40,659] Trial 2 finished with value: 0.03645322844386101 and parameters: {'units_1': 96, 'units_2': 48, 'dropout': 0.24466685667763072, 'learning_rate': 0.005006273985645114, 'batch_size': 32}. Best is trial 2 with value: 0.03645322844386101.
[I 2026-07-06 20:03:29,334] Trial 3 finished with value: 0.03746684640645981 and parameters: {'units_1': 32, 'units_2': 16, 'dropout': 0.34980915033745785, 'learning_rate': 0.0006641720247204887, 'batch_size': 

...

[I 2026-07-07 01:53:45,029] Trial 19 finished with value: 0.034698061645030975 and parameters: {'units_1': 64, 'units_2': 64, 'dropout': 0.4940859939329632, 'learning_rate': 0.00644874684902896, 'batch_size': 16}. Best is trial 16 with value: 0.028324265033006668.
Melhores hiperparâmetros | Ticker = MULT3 | window_size = 120
units_1: 64
units_2: 64
dropout: 0.30618407209730203
learning_rate: 0.008948811561232728
batch_size: 16

Melhor validation loss:
0.028324265033006668



Início dos estudos | Ticker = MULT3 | window_size = 180
(2289, 180, 21)
(2289,)
Treino     : (1602, 180, 21)
Validação  : (343, 180, 21)
Teste      : (344, 180, 21)


  0%|          | 0/20 [00:00<?, ?it/s]

g:\Meu Drive\5. Cursos\Pós ML Engineering\Fase 4 - Deep Learning e IA\lstm-stock-predictor-api\.venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


[I 2026-07-07 02:08:02,171] Trial 0 finished with value: 0.041008077561855316 and parameters: {'units_1': 128, 'units_2': 48, 'dropout': 0.23059270964959588, 'learning_rate': 0.00016001166348372277, 'batch_size': 64}. Best is trial 0 with value: 0.041008077561855316.
[I 2026-07-07 02:13:04,447] Trial 1 finished with value: 0.033530402928590775 and parameters: {'units_1': 96, 'units_2': 64, 'dropout': 0.19245011108453355, 'learning_rate': 0.0037045648853674643, 'batch_size': 32}. Best is trial 1 with value: 0.033530402928590775.
[I 2026-07-07 02:27:45,468] Trial 2 finished with value: 0.04407649114727974 and parameters: {'units_1': 128, 'units_2': 48, 'dropout': 0.2894340915186312, 'learning_rate': 0.00014468607135246285, 'batch_size': 64}. Best is trial 1 with value: 0.033530402928590775.
[I 2026-07-07 02:31:04,676] Trial 3 finished with value: 0.034433480352163315 and parameters: {'units_1': 128, 'units_2': 32, 'dropout': 0.4936499725977531, 'learning_rate': 0.002503108111619978, 'bat

OSError: [WinError 433] Foi especificado um dispositivo inexistente: 'dados\\aux_data'

In [6]:
tabela_hiperparams

,ticker,window_size,val_loss,units_1,units_2,dropout,learning_rate,batch_size
0,RENT3,60,0.06,96,16,0.10,0.01,32
0,RENT3,90,0.05,96,48,0.41,0.00,64
0,RENT3,120,0.03,96,48,0.20,0.01,16
0,RENT3,180,0.03,32,32,0.21,0.01,16
0,LREN3,60,0.00,64,16,0.50,0.00,16
0,LREN3,90,0.00,32,48,0.25,0.00,32
0,LREN3,120,0.00,32,32,0.17,0.00,64
0,LREN3,180,0.00,32,48,0.44,0.00,64


In [30]:
tabela_hiperparams

,ticker,window_size,val_loss,units_1,units_2,dropout,learning_rate,batch_size
0,MULT3,60,0.03,128,48,0.36,0.01,16
0,MULT3,90,0.03,64,32,0.23,0.01,16
0,MULT3,120,0.03,64,64,0.31,0.01,16
0,MULT3,180,0.03,32,48,0.16,0.01,32


In [39]:
# tabela_hiperparams.to_excel(r'C:\Users\Hydra\Downloads\tabela_hiperparametros_MULT3.xlsx', index = False)

In [6]:
tabela_hiperparams = pd.DataFrame()

df = pd.read_parquet(f"{save_folder_path_refined}/tb_analitica_VBBR3.parquet", engine = 'pyarrow')
df = df[df['close'].notna()]
df = df.sort_values("date").reset_index(drop=True)

data = df[features].copy()

scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(data)

for w in [60, 90, 120, 180]:
    print("=" * 60)
    print(f"Início dos estudos | Ticker = VBBR3 | window_size = {w}")
    print("=" * 60)

    study = get_study_optuna(w, scaled_data, n_trials = 20, ticker = 'VBBR3')

    tmp = pd.DataFrame(data = {
        'ticker':['VBBR3'],
        'window_size':[w],
        'val_loss':[study.best_value]

    })

    for k, v in study.best_params.items():
        tmp[k] = [v]

    tabela_hiperparams = pd.concat([tabela_hiperparams, tmp])
    print("\n\n")

tabela_hiperparams.to_excel(r'C:\Users\Hydra\Downloads\tabela_hiperparametros_VBBR3.xlsx', index = False)

[I 2026-07-07 07:05:54,671] A new study created in memory with name: LSTM Stock Prediction | window_size = 60


Início dos estudos | Ticker = VBBR3 | window_size = 60
(1980, 60, 21)
(1980,)
Treino     : (1386, 60, 21)
Validação  : (297, 60, 21)
Teste      : (297, 60, 21)


  0%|          | 0/20 [00:00<?, ?it/s]

g:\Meu Drive\5. Cursos\Pós ML Engineering\Fase 4 - Deep Learning e IA\lstm-stock-predictor-api\.venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


[I 2026-07-07 07:06:23,250] Trial 0 finished with value: 0.027280865237116814 and parameters: {'units_1': 64, 'units_2': 48, 'dropout': 0.14545538427105129, 'learning_rate': 0.0020704203495016675, 'batch_size': 64}. Best is trial 0 with value: 0.027280865237116814.
[I 2026-07-07 07:06:59,878] Trial 1 finished with value: 0.026996400207281113 and parameters: {'units_1': 32, 'units_2': 32, 'dropout': 0.25179175785805674, 'learning_rate': 0.0019192393913613252, 'batch_size': 32}. Best is trial 1 with value: 0.026996400207281113.
[I 2026-07-07 07:07:58,822] Trial 2 finished with value: 0.034790992736816406 and parameters: {'units_1': 32, 'units_2': 16, 'dropout': 0.3002621132155989, 'learning_rate': 0.00014805778551175267, 'batch_size': 64}. Best is trial 1 with value: 0.026996400207281113.
[I 2026-07-07 07:08:12,403] Trial 3 finished with value: 0.025059109553694725 and parameters: {'units_1': 96, 'units_2': 32, 'dropout': 0.2136487444962062, 'learning_rate': 0.007185847332193528, 'batch_

...

[I 2026-07-07 08:24:37,885] A new study created in memory with name: LSTM Stock Prediction | window_size = 180


[I 2026-07-07 08:24:37,873] Trial 19 finished with value: 0.021157091483473778 and parameters: {'units_1': 64, 'units_2': 16, 'dropout': 0.32429120239138653, 'learning_rate': 0.0035873493441501447, 'batch_size': 16}. Best is trial 9 with value: 0.019745392724871635.
Melhores hiperparâmetros | Ticker = VBBR3 | window_size = 120
units_1: 128
units_2: 64
dropout: 0.4039246030810061
learning_rate: 0.007080448271048392
batch_size: 16

Melhor validation loss:
0.019745392724871635



Início dos estudos | Ticker = VBBR3 | window_size = 180
(1860, 180, 21)
(1860,)
Treino     : (1302, 180, 21)
Validação  : (279, 180, 21)
Teste      : (279, 180, 21)


  0%|          | 0/20 [00:00<?, ?it/s]

g:\Meu Drive\5. Cursos\Pós ML Engineering\Fase 4 - Deep Learning e IA\lstm-stock-predictor-api\.venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


[I 2026-07-07 08:27:07,664] Trial 0 finished with value: 0.02073306404054165 and parameters: {'units_1': 96, 'units_2': 32, 'dropout': 0.461042151196278, 'learning_rate': 0.0014261522402190159, 'batch_size': 32}. Best is trial 0 with value: 0.02073306404054165.
[I 2026-07-07 08:30:02,125] Trial 1 finished with value: 0.020955529063940048 and parameters: {'units_1': 32, 'units_2': 32, 'dropout': 0.23976184091974592, 'learning_rate': 0.00044089234169939005, 'batch_size': 64}. Best is trial 0 with value: 0.02073306404054165.
[I 2026-07-07 08:30:53,093] Trial 2 finished with value: 0.019462214782834053 and parameters: {'units_1': 32, 'units_2': 16, 'dropout': 0.45406684415886134, 'learning_rate': 0.008244818771042361, 'batch_size': 32}. Best is trial 2 with value: 0.019462214782834053.
[I 2026-07-07 08:32:26,523] Trial 3 finished with value: 0.019859006628394127 and parameters: {'units_1': 32, 'units_2': 48, 'dropout': 0.409685374635242, 'learning_rate': 0.005021490464335611, 'batch_size':

## Avaliação e Definição

In [ ]:
def listar_arquivos_pasta(folder_path: str):

    files = []
    for (dirpath, dirnames, filenames) in walk(folder_path):
        files.extend(filenames)
        break

    return files

In [8]:
tb_abev = pd.DataFrame(
    data = {
        'ticker':'ABEV3',
        'window_size':[30, 60, 90, 120, 360],
        'val_loss':[0.00621827645227313, 0.005724024027585983, 0.005373380612581968, 0.004879966843873262, 0.00518820621073246], 
        'units_1':[64, 32, 128, 32, 32], 
        'units_2':[32, 16, 32, 16, 32], 
        'dropout':[0.29893700532625567, 0.21778495227726768, 0.3425725487095859, 0.3076547889480078, 0.4990013534433384],
        'learning_rate':[0.00015723169829245267, 0.0003489814349324155, 0.00020847351706535064, 0.0005856545376679272, 0.00035577683100581134], 
        'batch_size':[32, 64, 64, 64, 64] 
    }
)

In [9]:
tabela_hiperparams = pd.DataFrame()

for i in [i for i in listar_arquivos_pasta('dados/aux_data/') if i.endswith('.xlsx')]:
    tmp = pd.read_excel(f"dados/aux_data/{i}")
    tabela_hiperparams = pd.concat([tabela_hiperparams, tmp])

tabela_hiperparams = pd.concat([tabela_hiperparams, tb_abev])

tabela_hiperparams

,ticker,window_size,val_loss,units_1,units_2,dropout,learning_rate,batch_size
0,RENT3,60,0.06,96,16,0.10,0.01,32
1,RENT3,90,0.05,96,48,0.41,0.00,64
2,RENT3,120,0.03,96,48,0.20,0.01,16
3,RENT3,180,0.03,32,32,0.21,0.01,16
4,LREN3,60,0.00,64,16,0.50,0.00,16
5,LREN3,90,0.00,32,48,0.25,0.00,32
6,LREN3,120,0.00,32,32,0.17,0.00,64
7,LREN3,180,0.00,32,48,0.44,0.00,64
0,SMFT3,60,0.04,96,32,0.13,0.01,32
1,SMFT3,90,0.04,96,48,0.12,0.01,16


In [15]:
with pd.option_context('display.float_format', '{:,.5f}'.format):
    display(tabela_hiperparams.sort_values('val_loss').drop_duplicates(subset = 'ticker'))

,ticker,window_size,val_loss,units_1,units_2,dropout,learning_rate,batch_size
7,LREN3,180,0.00269,32,48,0.43652,0.00021,64
3,ABEV3,120,0.00488,32,16,0.30765,0.00059,64
3,VBBR3,180,0.01667,128,16,0.19183,0.00980,64
2,MULT3,120,0.02832,64,64,0.30618,0.00895,16
3,RENT3,180,0.03029,32,32,0.20686,0.00625,16
0,SMFT3,60,0.03602,96,32,0.12643,0.00920,32


In [17]:
tabela_hiperparams.sort_values('val_loss').drop_duplicates(subset = 'ticker').to_excel('dados/refined/tabela_melhores_hiperparametros.xlsx', index = False)

## 2º teste de hiperparametros

In [ ]:
for t in tickers:

    tabela_hiperparams = pd.DataFrame()

    df = pd.read_parquet(f"{save_folder_path_refined}/tb_analitica_{t}.parquet", engine = 'pyarrow')
    df = df[df['close'].notna()]
    df = df.sort_values("date").reset_index(drop=True)

    data = df[features].copy()

    # ==============================================================================
    # Split temporal do DataFrame
    # ==============================================================================

    n = len(data)

    train_end = int(n * 0.70)
    valid_end = int(n * 0.85)

    train_df = data.iloc[:train_end].copy()
    valid_df = data.iloc[train_end:valid_end].copy()
    test_df  = data.iloc[valid_end:].copy()

    # ==============================================================================
    # Normalização
    # O scaler é ajustado SOMENTE nos dados de treino
    # ==============================================================================

    scaler = RobustScaler()

    scaler.fit(train_df)

    train_scaled = scaler.transform(train_df)
    valid_scaled = scaler.transform(valid_df)
    test_scaled  = scaler.transform(test_df)

    # ==============================================================================
    # Construção das sequências
    # Cada conjunto gera suas próprias sequências
    # ==============================================================================

    print("=" * 60)
    print(f"Início dos estudos | Ticker = {t}")
    print("=" * 60)

    study = get_study_optuna(train_scaled, valid_scaled, test_scaled, n_trials = 50, ticker = t)

    tmp = pd.DataFrame(data = {
        'ticker':[t],
        'val_loss_mse':[study.best_value]
    })

    for k, v in study.best_params.items():
        tmp[k] = [v]

    tabela_hiperparams = pd.concat([tabela_hiperparams, tmp])
    print("\n\n")

    tabela_hiperparams.to_excel(f'dados/aux_data/tabela_hiperparametros_{t}.xlsx', index = False)